# 01 · Recolección de datos con la API REST de SpaceX

**Objetivo:** construir el dataset de lanzamientos del Falcon 9 a partir de la API pública de SpaceX
(`https://api.spacexdata.com/v4`).

**Flujo**

1. Descargar el histórico de lanzamientos. Igual que en el laboratorio de IBM, se usa la copia estática
   `API_call_spacex_api.json` para que los resultados sean reproducibles (la API en vivo cambia con el tiempo).
2. Los lanzamientos solo traen identificadores (`rocket`, `launchpad`, `payloads`, `cores`). Se resuelven
   consultando los endpoints `/rockets`, `/launchpads`, `/payloads` y `/cores`.
3. Filtrar misiones con un único núcleo y una única carga útil, y fechas hasta el 13-nov-2020.
4. Conservar solo el **Falcon 9**, imputar la masa de carga faltante con la media y guardar
   `data/dataset_part_1.csv`.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [ ]:
import datetime
import json

import numpy as np
import requests

pd.set_option("display.max_columns", None)

SPACEX_API = "https://api.spacexdata.com/v4"
STATIC_JSON_URL = f"{IBM}/datasets/API_call_spacex_api.json"
FALLBACK_CSV_URL = f"{IBM}/datasets/dataset_part_1.csv"

session = requests.Session()
session.headers["User-Agent"] = "ibm-capstone-spacex/1.0"

## 1. Petición de prueba a la API en vivo

Se comprueba que la API responde (código 200). El análisis usa la copia estática, pero esta celda demuestra
el uso de `requests` contra el endpoint real.

In [ ]:
try:
    live = session.get(f"{SPACEX_API}/launches/past", timeout=30)
    print("Código de estado de la API en vivo:", live.status_code)
except requests.RequestException as exc:
    print("La API en vivo no respondió:", exc)

## 2. Descarga del histórico de lanzamientos (copia estática del curso)

In [ ]:
response = session.get(STATIC_JSON_URL, timeout=60)
response.raise_for_status()
data = pd.json_normalize(response.json())
print(data.shape)
data.head()

## 3. Selección de columnas y filtros

* Solo misiones con **un** núcleo (`cores`) y **una** carga útil (`payloads`): las misiones con varios
  boosters (Falcon Heavy) o varias cargas complican el análisis.
* Fecha de corte: 13-nov-2020, igual que el laboratorio.

In [ ]:
data = data[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]]
data = data[data["cores"].map(len) == 1]
data = data[data["payloads"].map(len) == 1]
data["cores"] = data["cores"].map(lambda x: x[0])
data["payloads"] = data["payloads"].map(lambda x: x[0])
data["date"] = pd.to_datetime(data["date_utc"]).dt.date
data = data[data["date"] <= datetime.date(2020, 11, 13)]
print(data.shape)

## 4. Enriquecimiento con los endpoints de la API

Funciones auxiliares que, a partir de cada identificador, consultan la API y extraen los campos que
necesitamos. Se usa una caché en memoria para no repetir la misma petición.

In [ ]:
_cache: dict[str, dict] = {}


def get_json(endpoint: str, item_id: str) -> dict:
    key = f"{endpoint}/{item_id}"
    if key not in _cache:
        r = session.get(f"{SPACEX_API}/{key}", timeout=30)
        r.raise_for_status()
        _cache[key] = r.json()
    return _cache[key]


BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []


def getBoosterVersion(df):
    for rocket_id in df["rocket"]:
        BoosterVersion.append(get_json("rockets", rocket_id)["name"])


def getLaunchSite(df):
    for pad_id in df["launchpad"]:
        pad = get_json("launchpads", pad_id)
        Longitude.append(pad["longitude"])
        Latitude.append(pad["latitude"])
        LaunchSite.append(pad["name"])


def getPayloadData(df):
    for payload_id in df["payloads"]:
        payload = get_json("payloads", payload_id)
        PayloadMass.append(payload["mass_kg"])
        Orbit.append(payload["orbit"])


def getCoreData(df):
    for core in df["cores"]:
        if core["core"] is not None:
            info = get_json("cores", core["core"])
            Block.append(info["block"])
            ReusedCount.append(info["reuse_count"])
            Serial.append(info["serial"])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(f"{core['landing_success']} {core['landing_type']}")
        Flights.append(core["flight"])
        GridFins.append(core["gridfins"])
        Reused.append(core["reused"])
        Legs.append(core["legs"])
        LandingPad.append(core["landpad"])

In [ ]:
try:
    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)
    api_ok = True
except requests.RequestException as exc:
    api_ok = False
    print("No se pudo completar el enriquecimiento con la API:", exc)

## 5. Construcción del DataFrame

In [ ]:
if api_ok:
    launch_dict = {
        "FlightNumber": list(data["flight_number"]),
        "Date": list(data["date"]),
        "BoosterVersion": BoosterVersion,
        "PayloadMass": PayloadMass,
        "Orbit": Orbit,
        "LaunchSite": LaunchSite,
        "Outcome": Outcome,
        "Flights": Flights,
        "GridFins": GridFins,
        "Reused": Reused,
        "Legs": Legs,
        "LandingPad": LandingPad,
        "Block": Block,
        "ReusedCount": ReusedCount,
        "Serial": Serial,
        "Longitude": Longitude,
        "Latitude": Latitude,
    }
    launch_df = pd.DataFrame(launch_dict)
    source = "SpaceX REST API v4"
else:
    # Respaldo documentado: el dataset_part_1.csv que publica IBM es el resultado de este mismo proceso.
    print("Usando el respaldo oficial del curso:", FALLBACK_CSV_URL)
    launch_df = pd.read_csv(FALLBACK_CSV_URL)
    source = "IBM dataset_part_1.csv (respaldo)"
launch_df.head()

## 6. Filtrar solo Falcon 9 y renumerar los vuelos

In [ ]:
data_falcon9 = launch_df[launch_df["BoosterVersion"] != "Falcon 1"].copy()
data_falcon9["FlightNumber"] = list(range(1, data_falcon9.shape[0] + 1))
print("Lanzamientos Falcon 9:", len(data_falcon9))
data_falcon9.isnull().sum()

## 7. Tratamiento de valores faltantes

`PayloadMass` tiene valores nulos: se reemplazan por la **media**. `LandingPad` conserva sus nulos, porque
significan "no se usó plataforma de aterrizaje" y esa ausencia es información.

In [ ]:
payload_mean = data_falcon9["PayloadMass"].astype(float).mean()
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].astype(float).fillna(payload_mean)
print(f"Media usada para imputar PayloadMass: {payload_mean:,.2f} kg")
data_falcon9.isnull().sum()

In [ ]:
data_falcon9.to_csv(DATA_DIR / "dataset_part_1.csv", index=False)

summary = {
    "source": source,
    "rows": int(len(data_falcon9)),
    "columns": list(data_falcon9.columns),
    "payload_mean_imputed_kg": round(float(payload_mean), 2),
}
(RESULTS_DIR / "data_collection_api.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

## Conclusión

Se obtuvo un dataset limpio de lanzamientos Falcon 9 con 17 variables (versión del booster, masa de carga,
órbita, sitio, resultado del aterrizaje, reutilización, patas, aletas de rejilla, etc.), listo para el
*data wrangling* del notebook 03.